In [ ]:
!pip -q install sentence-transformers scikit-learn numpy pandas

In [ ]:
import json
import random
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

get all the json files in a similar format

In [ ]:
# ---------------------------
# config
# ---------------------------

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

YELLOW_JSON = "/content/Yellow Data.json"
GREEN_JSON  = "/content/green_categories.json"
BLUE_JSON   = "/content/blue_trivia.json"
PURPLE_JSON = "/content/purple_categories.json"

# how many categories to sample from each color START SMALL FOR TRIAL
SAMPLE_PER_COLOR = 200

# number of clusters for the trial
# for 1000 categories total, 80-140 is a decent trial band
N_CLUSTERS = 100

# random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# ---------------------------
# helpers
# ---------------------------

def norm_text(s):
    # normalize text for consistency
    return str(s).strip().upper()


@dataclass
class Category:
    # one normalized category record
    color: str
    group: str
    members: list
    cat_type: str = "unknown"
    explanation: str = ""
    source_index: int = -1

    # fields filled later
    label_vec = None
    member_vecs = None
    member_centroid = None
    combo_vec = None
    hybrid_vec = None
    cluster_id = None

    def display(self):
        return {
            "color": self.color.upper(),
            "group": self.group.upper(),
            "members": [norm_text(x) for x in self.members],
            "cat_type": self.cat_type,
            "explanation": self.explanation,
            "source_index": self.source_index,
            "cluster_id": self.cluster_id,
        }


def load_categories_schema_aware(path, color):
    # load one of your current json formats and normalize it
    raw = json.load(open(path, "r", encoding="utf-8"))
    out = []

    for i, item in enumerate(raw):
        group = None
        members = None
        cat_type = "unknown"
        explanation = ""

        if color == "blue":
            # blue schema
            # { "group": "...", "members": [...] }
            group = item.get("group")
            members = item.get("members")
            cat_type = item.get("type", "blue")

        elif color == "green":
            # green schema
            # { "type": "...", "label": "...", "words": [...] }
            group = item.get("label")
            members = item.get("words")
            cat_type = item.get("type", "green")

        elif color == "yellow":
            # yellow schema
            # { "id": 0, "category": "...", "difficulty": "yellow", "words": [...] }
            group = item.get("category")
            members = item.get("words")
            cat_type = item.get("difficulty", "yellow")

        elif color == "purple":
            # purple schema
            # { "group": "...", "members": [...], "explanation": "...", "type": "..." }
            group = item.get("group")
            members = item.get("members")
            explanation = item.get("explanation", "")
            cat_type = item.get("type", "purple")

        else:
            raise ValueError(f"unknown color: {color}")

        # keep only proper 4-word categories
        if not group or not members or len(members) != 4:
            continue

        out.append(
            Category(
                color=color,
                group=norm_text(group),
                members=[norm_text(m) for m in members],
                cat_type=cat_type,
                explanation=explanation,
                source_index=i,
            )
        )

    return out

In [ ]:
# ---------------------------
# load json files
# ---------------------------

yellow_all = load_categories_schema_aware(YELLOW_JSON, "yellow")
green_all  = load_categories_schema_aware(GREEN_JSON, "green")
blue_all   = load_categories_schema_aware(BLUE_JSON, "blue")
purple_all = load_categories_schema_aware(PURPLE_JSON, "purple")

print("loaded counts")
print("yellow:", len(yellow_all))
print("green :", len(green_all))
print("blue  :", len(blue_all))
print("purple:", len(purple_all))

loaded counts
yellow: 2187
green : 13925
blue  : 2000
purple: 3248


In [ ]:
# ---------------------------
# sample a small random trial set
# ---------------------------

def sample_categories(cats, n, seed=None):
    # sample up to n categories from one bank
    if seed is not None:
        random.seed(seed)
    n = min(n, len(cats))
    return random.sample(cats, n)

yellow_sample = sample_categories(yellow_all, SAMPLE_PER_COLOR, seed=RANDOM_SEED + 1)
green_sample  = sample_categories(green_all,  SAMPLE_PER_COLOR, seed=RANDOM_SEED + 2)
blue_sample   = sample_categories(blue_all,   SAMPLE_PER_COLOR, seed=RANDOM_SEED + 3)
purple_sample = sample_categories(purple_all, SAMPLE_PER_COLOR, seed=RANDOM_SEED + 4)

trial_cats = yellow_sample + green_sample + blue_sample + purple_sample

print("\ntrial counts")
print("yellow:", len(yellow_sample))
print("green :", len(green_sample))
print("blue  :", len(blue_sample))
print("purple:", len(purple_sample))
print("total :", len(trial_cats))


trial counts
yellow: 200
green : 200
blue  : 200
purple: 200
total : 800


In [ ]:
# ---------------------------
# embedding helpers
# ---------------------------

def normalize_vec(v):
    # normalize one vector to unit length
    n = np.linalg.norm(v)
    if n == 0:
        return v
    return v / n


def centroid(vecs):
    # average vectors and renormalize
    c = np.mean(vecs, axis=0)
    return normalize_vec(c)


def embed_texts(model, texts):
    # encode a list of texts into normalized embeddings
    return model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )


def preprocess_categories(model, cats):
    # build multiple embeddings for every category:
    # 1) label only
    # 2) the 4 words
    # 3) a combined text like "GROUP: A, B, C, D"
    # 4) a blended hybrid vector for clustering

    label_texts = [c.group for c in cats]
    combo_texts = [f"{c.group}: {', '.join(c.members)}" for c in cats]

    label_vecs = embed_texts(model, label_texts)
    combo_vecs = embed_texts(model, combo_texts)

    all_words = []
    spans = []
    start = 0

    for c in cats:
        all_words.extend(c.members)
        spans.append((start, start + 4))
        start += 4

    all_word_vecs = embed_texts(model, all_words)

    for i, c in enumerate(cats):
        lo, hi = spans[i]

        c.label_vec = label_vecs[i]
        c.member_vecs = all_word_vecs[lo:hi]
        c.member_centroid = centroid(c.member_vecs)
        c.combo_vec = combo_vecs[i]

        # hybrid embedding
        # this blend works fairly well across your mixed schemas
        c.hybrid_vec = centroid(np.vstack([
            0.55 * c.combo_vec,
            0.30 * c.member_centroid,
            0.15 * c.label_vec
        ]))

In [ ]:
# ---------------------------
# run embeddings
# ---------------------------

model = SentenceTransformer(MODEL_NAME)
preprocess_categories(model, trial_cats)

print("embeddings ready for", len(trial_cats), "categories")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embeddings ready for 800 categories


In [ ]:
# ---------------------------
# clustering
# ---------------------------

def build_embedding_matrix(cats):
    # stack all hybrid vectors into one matrix
    return np.vstack([c.hybrid_vec for c in cats])


def cluster_categories(cats, n_clusters=100):
    # cluster categories using agglomerative clustering on cosine distance
    # because vectors are already normalized, cosine-based grouping works well

    X = build_embedding_matrix(cats)

    clustering = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="cosine",
        linkage="average"
    )

    labels = clustering.fit_predict(X)

    for c, label in zip(cats, labels):
        c.cluster_id = int(label)

    return labels

In [ ]:
# ---------------------------
# run clustering
# ---------------------------

cluster_labels = cluster_categories(trial_cats, n_clusters=N_CLUSTERS)
print("done clustering")
print("unique clusters:", len(set(cluster_labels)))

done clustering
unique clusters: 100


In [ ]:
# ---------------------------
# cluster inspection helpers
# ---------------------------

def group_by_cluster(cats):
    # map cluster id -> list of categories
    clusters = defaultdict(list)
    for c in cats:
        clusters[c.cluster_id].append(c)
    return clusters


def cluster_color_mix(cluster_cats):
    # count colors inside one cluster
    return Counter(c.color for c in cluster_cats)


def cluster_centroid(cluster_cats):
    # centroid of all category embeddings in one cluster
    vecs = np.vstack([c.hybrid_vec for c in cluster_cats])
    return centroid(vecs)


def average_similarity_to_cluster(cluster_cats):
    # measure how tight a cluster is
    if len(cluster_cats) < 2:
        return 1.0

    X = np.vstack([c.hybrid_vec for c in cluster_cats])
    sim = cosine_similarity(X)
    vals = []

    for i in range(len(cluster_cats)):
        for j in range(i + 1, len(cluster_cats)):
            vals.append(sim[i, j])

    if not vals:
        return 1.0

    return float(np.mean(vals))


def summarize_clusters(cats):
    # build a dataframe summary of all clusters
    clusters = group_by_cluster(cats)
    rows = []

    for cid, members in clusters.items():
        mix = cluster_color_mix(members)
        rows.append({
            "cluster_id": cid,
            "size": len(members),
            "avg_similarity": average_similarity_to_cluster(members),
            "yellow": mix.get("yellow", 0),
            "green":  mix.get("green", 0),
            "blue":   mix.get("blue", 0),
            "purple": mix.get("purple", 0),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(
        ["size", "avg_similarity"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return df

In [ ]:
cluster_df = summarize_clusters(trial_cats)
cluster_df.head(20)

,cluster_id,size,avg_similarity,yellow,green,blue,purple
0,23,73,0.441030,0,0,0,73
1,14,57,0.455984,0,1,11,45
2,32,48,0.437047,0,1,0,47
3,13,37,0.417554,0,37,0,0
4,21,27,0.448328,11,6,10,0
5,88,25,0.520081,1,0,24,0
6,5,25,0.486515,2,3,20,0
7,10,22,0.431236,0,0,0,22
8,19,21,0.524860,2,2,17,0
9,27,18,0.519851,4,2,12,0


In [ ]:
# ---------------------------
# pretty printing clusters
# ---------------------------

def print_cluster(cluster_id, cats, max_items=20):
    # show categories inside one cluster
    clusters = group_by_cluster(cats)
    members = clusters.get(cluster_id, [])

    print(f"\n=== CLUSTER {cluster_id} ===")
    print("size:", len(members))
    print("avg similarity:", round(average_similarity_to_cluster(members), 4))
    print("color mix:", dict(cluster_color_mix(members)))

    # sort by color first so the cluster is easier to read
    members = sorted(members, key=lambda c: (c.color, c.group))

    for i, c in enumerate(members[:max_items], 1):
        print(f"\n[{i}] {c.color.upper()} | {c.group}")
        print("    " + ", ".join(c.members))
        print("    type:", c.cat_type)
        if c.explanation:
            print("    explanation:", c.explanation)

    if len(members) > max_items:
        print(f"\n... and {len(members) - max_items} more")

In [ ]:
# show one of the biggest clusters
top_cluster_id = int(cluster_df.iloc[0]["cluster_id"])
print_cluster(top_cluster_id, trial_cats, max_items=25)


=== CLUSTER 23 ===
size: 73
avg similarity: 0.441
color mix: {'purple': 73}

[1] PURPLE | ADD "S" TO THE FRONT = NEW WORD
    PACE, HARP, MILE, PIT
    type: meta_s_front
    explanation: S+PACE = SPACE, S+HARP = SHARP, S+MILE = SMILE, S+PIT = SPIT

[2] PURPLE | ADD "S" TO THE FRONT = NEW WORD
    QUID, CUM, QUAD, HEAR
    type: meta_s_front
    explanation: S+QUID = SQUID, S+CUM = SCUM, S+QUAD = SQUAD, S+HEAR = SHEAR

[3] PURPLE | ADD "S" TO THE FRONT = NEW WORD
    TOW, HIV, TUN, NAPPY
    type: meta_s_front
    explanation: S+TOW = STOW, S+HIV = SHIV, S+TUN = STUN, S+NAPPY = SNAPPY

[4] PURPLE | EACH MINUS A LETTER = "LEA"
    LEAR, LENA, LEAD, LEAN
    type: letter_drop
    explanation: LEAR -> LEA, LENA -> LEA, LEAD -> LEA, LEAN -> LEA

[5] PURPLE | EACH MINUS A LETTER = "ROT"
    RIOT, ROOT, TROT, ROTH
    type: letter_drop
    explanation: RIOT -> ROT, ROOT -> ROT, TROT -> ROT, ROTH -> ROT

[6] PURPLE | ENDING WORDS THAT RHYME
    FLAUNT, MIGRANT, SERVANT, OCCUPANT
    type: tw

In [ ]:
# ---------------------------
# mixed-color clusters
# ---------------------------

def count_nonzero_colors(row):
    # number of colors represented in a cluster
    return sum([
        row["yellow"] > 0,
        row["green"] > 0,
        row["blue"] > 0,
        row["purple"] > 0,
    ])

cluster_df["num_colors"] = cluster_df.apply(count_nonzero_colors, axis=1)

mixed_clusters = cluster_df[cluster_df["num_colors"] >= 3].copy()
mixed_clusters = mixed_clusters.sort_values(
    ["num_colors", "size", "avg_similarity"],
    ascending=[False, False, False]
)

mixed_clusters.head(20)

,cluster_id,size,avg_similarity,yellow,green,blue,purple,num_colors
1,14,57,0.455984,0,1,11,45,3
4,21,27,0.448328,11,6,10,0,3
6,5,25,0.486515,2,3,20,0,3
8,19,21,0.524860,2,2,17,0,3
9,27,18,0.519851,4,2,12,0,3
10,4,18,0.475300,1,2,15,0,3
11,17,16,0.446176,11,2,3,0,3
12,35,13,0.527138,3,2,8,0,3
14,7,11,0.512437,1,1,9,0,3
15,8,11,0.467204,3,3,5,0,3


In [ ]:
# inspect a strong mixed-color cluster
if len(mixed_clusters) > 0:
    cid = int(mixed_clusters.iloc[0]["cluster_id"])
    print_cluster(cid, trial_cats, max_items=30)


=== CLUSTER 14 ===
size: 57
avg similarity: 0.456
color mix: {'green': 1, 'blue': 11, 'purple': 45}

[1] BLUE | MEMBERS OF THE BROADER CATEGORY OF GREEK LANGUAGE
    IOTACISM, CRASIS, KAI, OSTRACON
    type: blue

[2] BLUE | MEMBERS OF THE BROADER CATEGORY OF JAPANESE LANGUAGE
    MANGAJIN, CHASEN, YAKUWARIGO, SINICIZATION
    type: blue

[3] BLUE | MEMBERS OF THE BROADER CATEGORY OF LINGUISTICS
    PSEUDOWORD, LEXICON, FILLER, METALINGUISTICS
    type: blue

[4] BLUE | MEMBERS OF THE BROADER CATEGORY OF LINGUISTICS
    PSEUDOWORD, LEXICON, FILLER, METALINGUISTICS
    type: blue

[5] BLUE | MEMBERS OF THE BROADER CATEGORY OF LINGUISTICS
    PSEUDOWORD, LEXICON, FILLER, METALINGUISTICS
    type: blue

[6] BLUE | MEMBERS OF THE BROADER CATEGORY OF LINGUISTICS
    PSEUDOWORD, LEXICON, FILLER, METALINGUISTICS
    type: blue

[7] BLUE | MEMBERS OF THE BROADER CATEGORY OF LINGUISTICS
    PSEUDOWORD, LEXICON, FILLER, METALINGUISTICS
    type: blue

[8] BLUE | MEMBERS OF THE BROADER CATEGORY 

In [ ]:
# ---------------------------
# nearest cross-color neighbors
# ---------------------------

def nearest_cross_color_neighbors(query_cat, candidate_cats, top_k=10):
    # find nearest categories from other colors
    rows = []

    for c in candidate_cats:
        if c.color == query_cat.color:
            continue

        sim = float(np.dot(query_cat.hybrid_vec, c.hybrid_vec))
        rows.append({
            "query_color": query_cat.color,
            "query_group": query_cat.group,
            "candidate_color": c.color,
            "candidate_group": c.group,
            "candidate_words": ", ".join(c.members),
            "similarity": sim,
            "candidate_type": c.cat_type,
        })

    df = pd.DataFrame(rows)
    df = df.sort_values("similarity", ascending=False).reset_index(drop=True)
    return df.head(top_k)

In [ ]:
# try it on one random purple category
random_purple = random.choice([c for c in trial_cats if c.color == "purple"])
print("QUERY PURPLE")
print(random_purple.group)
print(", ".join(random_purple.members))

nearest_cross_color_neighbors(random_purple, trial_cats, top_k=15)

QUERY PURPLE
TWO SETS OF DOUBLE LETTERS
ACCESSION, ACCIDENTALLY, ADDITIONALLY, XXIII


,query_color,query_group,candidate_color,candidate_group,candidate_words,similarity,candidate_type
0,purple,TWO SETS OF DOUBLE LETTERS,green,SIBLINGS UNDER OFFER,"BID, PROPOSITION, PROSPECTUS, SPECIAL",0.405218,sister_hyponyms
1,purple,TWO SETS OF DOUBLE LETTERS,green,TYPES OF EQUIVALENT,"COMPLEMENT, MISMATCH, REPLACEMENT, SUCCESSOR",0.377998,types_of
2,purple,TWO SETS OF DOUBLE LETTERS,yellow,CHANGE OF INTEGRITY TERMS,"BURNING, DAMAGE, HARM, HURT",0.361912,yellow
3,purple,TWO SETS OF DOUBLE LETTERS,green,SIBLINGS UNDER FLAP,"FLY, JAG, LAP, TONGUE",0.354168,sister_hyponyms
4,purple,TWO SETS OF DOUBLE LETTERS,green,SIBLINGS UNDER COMBINATION,"BLEND, MIX, TEMPERANCE, UNION",0.354142,sister_hyponyms
5,purple,TWO SETS OF DOUBLE LETTERS,yellow,IMPROVEMENT TERMS,"CLEARING, CORRECTION, FIX, FIXING",0.353678,yellow
6,purple,TWO SETS OF DOUBLE LETTERS,green,HAVING OR DEMONSTRATING ABILITY TO RECOGNIZE O...,"ACUTE, PENETRATING, PIERCING, SHARP",0.352950,synonyms
7,purple,TWO SETS OF DOUBLE LETTERS,yellow,TYPES OF RIVALS,"CHAMPION, FAVORITE, FAVOURITE, SCRATCH",0.345257,yellow
8,purple,TWO SETS OF DOUBLE LETTERS,green,SIBLINGS UNDER IMPROVEMENT,"CORRECTION, ENHANCEMENT, RENOVATION, UPGRADE",0.344961,sister_hyponyms
9,purple,TWO SETS OF DOUBLE LETTERS,green,SIBLINGS UNDER CUT,"JOINT, NECK, RIB, SHANK",0.338350,sister_hyponyms
